### **Objective of This Notebook**
The purpose of this notebook is to transform the structurally validated dataset produced during the exploratory analysis phase into a clean and modelling-ready dataset. This includes:
*   consolidating duplicated tracks
*   handling missing values
*   engineering multi-label genre features
*   preparing feature groups
*   and exporting the final processed dataset for predictive modelling







# **1. Imports and Config**

In [8]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import MultiLabelBinarizer
import requests

### **1.1 Notebook Configuration**

In [9]:
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.3f}".format)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

### **1.2 Paths**

In [10]:
PROJECT_ROOT = Path("/content/spotify-popularity-prediction")
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "dataset.csv"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"
FIGURES_PATH = PROJECT_ROOT / "outputs" / "figures" / "eda"

RAW_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)
FIGURES_PATH.mkdir(parents=True, exist_ok=True)

### **1.3 Save into DataFrame**

In [11]:
df = pd.read_csv(RAW_DATA_PATH)
df = df.drop(columns=["Unnamed: 0"])

print("Loaded raw dataset:", df.shape)

FileNotFoundError: [Errno 2] No such file or directory: '/content/spotify-popularity-prediction/data/raw/dataset.csv'

# **2. Initial Cleanup**

### **2.1 Aggregate and consolidate track genres**

In [ ]:
# =========================================================
# Aggregate genres and consolidate duplicated tracks
# =========================================================

aggregated_df = (
    df.groupby("track_id")
    .agg({
        "artists": "first",
        "album_name": "first",
        "track_name": "first",
        "popularity": "first",
        "duration_ms": "first",
        "explicit": "first",
        "danceability": "first",
        "energy": "first",
        "key": "first",
        "loudness": "first",
        "mode": "first",
        "speechiness": "first",
        "acousticness": "first",
        "instrumentalness": "first",
        "liveness": "first",
        "valence": "first",
        "tempo": "first",
        "time_signature": "first",

        # Concatenate unique genres
        "track_genre": lambda x: ", ".join(
            sorted(set(x))
        )
    })
    .reset_index()
)

In [ ]:
print(f"Original shape: {df.shape}")
print(f"Aggregated shape: {aggregated_df.shape}")

In [ ]:
aggregated_df["track_id"].duplicated().sum()

Let's inspect track_genre was aggregated correctly

In [ ]:
aggregated_df[
    ["track_name", "artists", "track_genre"]
].head(10)

### **2.2 Drop missing values**

In [ ]:
aggregated_df = aggregated_df.dropna()

Save aggregated_df

In [ ]:
agg_path = PROCESSED_PATH / "aggregated_df.csv"
aggregated_df.to_csv(agg_path, index=False)
print("Saved aggregated dataframe to:", agg_path)


# **3. Feature engineering**

### **3.1 Create feature type lists**

In [ ]:
numerical_features = [
    "duration_ms",
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo"
]

binary_features = [
    "explicit",
    "mode"
]

categorical_features = [
    "key",
    "time_signature"
]

In [ ]:
aggregated_df["track_genre"] = (
    aggregated_df["track_genre"]
    .str.split(", ")
)

### **3.2 Multi-hot Encode**

In [ ]:
mlb = MultiLabelBinarizer()

genre_encoded = pd.DataFrame(
    mlb.fit_transform(
        aggregated_df["track_genre"]
    ),
    columns=mlb.classes_,
    index=aggregated_df.index
)

In [ ]:
aggregated_df = pd.concat(
    [aggregated_df, genre_encoded],
    axis=1
)

In [ ]:
aggregated_df = aggregated_df.drop(
    columns=["track_genre"]
)

Drop original track_genre as it's not needed anymore

### **3.3 Final dataset inspection**

In [ ]:
aggregated_df.head(20)

In [ ]:
aggregated_df.info()

In [ ]:
aggregated_df.head()

In [ ]:
aggregated_df.shape

In [ ]:
!ls /content/spotify-popularity-prediction/outputs/figures/eda